# Test functions for loading without MuData

In [74]:
from pathlib import Path

import torch
import numpy as np

import mudata as md
import pyro

from scipy.sparse import random as sparse_random
from scipy.sparse import csr_matrix, vstack
from scipy.stats import nbinom

import perturbo

import os
os.getcwd()

'/ictstr01/home/icb/jiaqi.lu/work/proj0.5_ProbGen/PerTurbo/power/power_cost_by_dataset/streamlit'

In [2]:
model_dir = Path("save_model/model_no_data_with_libsize")
print(f"Model is loaded from path: {model_dir}")

ckpt = torch.load(model_dir / "model.pt", map_location="cpu")
registry = ckpt["attr_dict"]["registry_"]
print(f"Registry {registry} elements.")

Model is loaded from path: save_model/model_no_data_with_libsize
Registry {'scvi_version': '1.4.0.post1', 'model_name': 'PERTURBO', 'setup_args': {'rna_layer': None, 'perturbation_layer': None, 'batch_key': 'prep_batch', 'gene_by_element_key': 'element_tested', 'rna_element_uns_key': None, 'guide_element_uns_key': 'elements', 'guide_by_element_key': 'element_targeted', 'library_size_key': 'umi_count', 'size_factor_key': None, 'gene_mean_key': None, 'continuous_covariates_keys': ['percent_mito', 'log1p_guide_count'], 'categorical_covariates_keys': None, 'modalities': {'rna_layer': 'rna', 'perturbation_layer': 'grna'}}, 'field_registries': defaultdict(<class 'dict'>, {'ind_x': {'data_registry': {'attr_name': 'obs', 'attr_key': '_ind_x', 'mod_key': 'rna'}, 'state_registry': {}, 'summary_stats': {}}, 'batch': {'data_registry': {'attr_name': 'obs', 'attr_key': '_scvi_batch', 'mod_key': 'rna'}, 'state_registry': {'categorical_mapping': array(['prep_batch_1', 'prep_batch_2'], dtype=object), '

## Test load_resource() function ------- WRONG WAY!

In [3]:
import inspect
inspect.signature(perturbo.PERTURBO).parameters

mappingproxy({'mdata': <Parameter "mdata: anndata._core.anndata.AnnData | mudata._core.mudata.MuData">,
              'registry': <Parameter "registry: dict = None">,
              'control_guides': <Parameter "control_guides: list | None = None">,
              'dispersion_smoothing': <Parameter "dispersion_smoothing: str = 'none'">,
              'smoothing_factor': <Parameter "smoothing_factor: float = 0.3">,
              'model_kwargs': <Parameter "**model_kwargs">})

In [5]:
from scvi.model.base._save_load import _load_saved_files, _initialize_model

# 1) Read saved attrs/state without loading AnnData from disk
attr_dict, var_names, state_dict, new_adata = _load_saved_files(
    model_dir, load_adata=False, map_location="cpu"
)

# 2) Extract the registry for initialization
registry = attr_dict.pop("registry_")
print(f"Registry {registry} elements.")

# 3) Build the model instance the same way scvi does
#model = _initialize_model(perturbo.PERTURBO, adata=None, registry=registry, attr_dict=attr_dict, datamodule=None)

INFO     File save_model/model_no_data/model.pt already downloaded                                                 
Registry {'scvi_version': '1.4.0.post1', 'model_name': 'PERTURBO', 'setup_args': {'rna_layer': None, 'perturbation_layer': None, 'batch_key': 'prep_batch', 'gene_by_element_key': 'element_tested', 'rna_element_uns_key': None, 'guide_element_uns_key': 'elements', 'guide_by_element_key': 'element_targeted', 'library_size_key': None, 'size_factor_key': None, 'gene_mean_key': None, 'continuous_covariates_keys': ['percent_mito', 'log1p_guide_count'], 'categorical_covariates_keys': None, 'modalities': {'rna_layer': 'rna', 'perturbation_layer': 'grna'}}, 'field_registries': defaultdict(<class 'dict'>, {'ind_x': {'data_registry': {'attr_name': 'obs', 'attr_key': '_ind_x', 'mod_key': 'rna'}, 'state_registry': {}, 'summary_stats': {}}, 'batch': {'data_registry': {'attr_name': 'obs', 'attr_key': '_scvi_batch', 'mod_key': 'rna'}, 'state_registry': {'categorical_mapping': array(['prep

What is inside ```registry```

In [23]:
def show_structure(obj, indent=0, max_list=5):
    pad = "  " * indent
    if isinstance(obj, dict):
        print(f"{pad}dict[{len(obj)}]")
        for k, v in obj.items():
            t = type(v).__name__
            print(f"{pad}├─ {k}: {t}")
            if isinstance(v, (dict, list)):
                show_structure(v, indent+1, max_list)
    elif isinstance(obj, list):
        print(f"{pad}list[{len(obj)}]")
        for i, item in enumerate(obj[:max_list]):
            t = type(item).__name__
            print(f"{pad}├─ [{i}]: {t}")
            if isinstance(item, (dict, list)):
                show_structure(item, indent+1, max_list)
        if len(obj) > max_list:
            print(f"{pad}└ … ({len(obj)-max_list} more)")
show_structure(registry)

dict[6]
├─ scvi_version: str
├─ model_name: str
├─ setup_args: dict
  dict[13]
  ├─ rna_layer: NoneType
  ├─ perturbation_layer: NoneType
  ├─ batch_key: str
  ├─ gene_by_element_key: str
  ├─ rna_element_uns_key: NoneType
  ├─ guide_element_uns_key: str
  ├─ guide_by_element_key: str
  ├─ library_size_key: NoneType
  ├─ size_factor_key: NoneType
  ├─ gene_mean_key: NoneType
  ├─ continuous_covariates_keys: list
    list[2]
    ├─ [0]: str
    ├─ [1]: str
  ├─ categorical_covariates_keys: NoneType
  ├─ modalities: dict
    dict[2]
    ├─ rna_layer: str
    ├─ perturbation_layer: str
├─ field_registries: defaultdict
  dict[8]
  ├─ ind_x: dict
    dict[3]
    ├─ data_registry: dict
      dict[3]
      ├─ attr_name: str
      ├─ attr_key: str
      ├─ mod_key: str
    ├─ state_registry: dict
      dict[0]
    ├─ summary_stats: dict
      dict[0]
  ├─ batch: dict
    dict[3]
    ├─ data_registry: dict
      dict[3]
      ├─ attr_name: str
      ├─ attr_key: str
      ├─ mod_key: str
    ├─

In [12]:
registry.keys()

dict_keys(['scvi_version', 'model_name', 'setup_args', 'field_registries', 'setup_method_name', '_scvi_uuid'])

In [39]:
registry['scvi_version']

'1.4.0.post1'

In [10]:
registry['field_registries'].keys()

dict_keys(['ind_x', 'batch', 'perturbations', 'X', 'size_factor', 'extra_continuous_covs', 'tested_elements', 'targeted_elements'])

In [10]:
registry['field_registries']['extra_continuous_covs']

{'data_registry': {'attr_name': 'obsm',
  'attr_key': '_scvi_extra_continuous_covs',
  'mod_key': 'rna'},
 'state_registry': {'columns': array(['percent_mito', 'log1p_guide_count'], dtype=object)},
 'summary_stats': {'n_extra_continuous_covs': 2}}

What is in torch.load(model.pt) ckpt?

In [43]:
ckpt

{'model_state_dict': OrderedDict([('guide_by_element',
               tensor(indices=tensor([[    0,     1,     2,  ..., 13186, 13187, 13188],
                                      [ 6585,  6588,  6590,  ...,   379,   380,   380]]),
                      values=tensor([1., 1., 1.,  ..., 1., 1., 1.]),
                      size=(13189, 6598), nnz=13189, layout=torch.sparse_coo)),
              ('element_by_gene',
               tensor(indices=tensor([[    0,     0,     0,  ...,  6597,  6597,  6597],
                                      [ 4730,  4731,  4732,  ..., 13113, 13114, 13115]]),
                      values=tensor([1., 1., 1.,  ..., 1., 1., 1.]),
                      size=(6598, 13135), nnz=629515, layout=torch.sparse_coo)),
              ('element_by_gene_idx',
               tensor([[    0,     0,     0,  ...,  6597,  6597,  6597],
                       [ 4730,  4731,  4732,  ..., 13113, 13114, 13115]])),
              ('guide_by_gene_idx',
               tensor([[    0,   

In [44]:
show_structure(ckpt)

dict[3]
├─ model_state_dict: OrderedDict
  dict[38]
  ├─ guide_by_element: Tensor
  ├─ element_by_gene: Tensor
  ├─ element_by_gene_idx: Tensor
  ├─ guide_by_gene_idx: Tensor
  ├─ zero: Tensor
  ├─ one: Tensor
  ├─ gene_mean_prior_loc: Tensor
  ├─ gene_disp_prior_loc: Tensor
  ├─ gene_mean_prior_scale: Tensor
  ├─ gene_disp_prior_scale: Tensor
  ├─ batch_effect_prior_scale: Tensor
  ├─ covariate_prior_sigma: Tensor
  ├─ covariate_disp_prior_sigma: Tensor
  ├─ logit_efficacy_alpha: Tensor
  ├─ logit_efficacy_beta: Tensor
  ├─ has_guide_prior: Tensor
  ├─ element_effects_prior_scale: Tensor
  ├─ guide_effects_prior_scale: Tensor
  ├─ spike_slab_prior_scales: Tensor
  ├─ spike_slab_prior_probs: Tensor
  ├─ cell_factor_prior_scale: Tensor
  ├─ cell_loading_prior_scale: Tensor
  ├─ pert_factor_prior_scale: Tensor
  ├─ pert_loading_prior_scale: Tensor
  ├─ noise_prior_rate: Tensor
  ├─ _guide.0.locs.element_effects_unconstrained: Tensor
  ├─ _guide.0.locs.guide_efficacy_unconstrained: Tensor

In [51]:
ckpt["attr_dict"].keys()

dict_keys(['get_normalized_function_name_', 'history_', 'init_params_', 'is_trained_', 'registry_', 'test_indices_', 'train_indices_', 'validation_indices_'])

In [49]:
ckpt["attr_dict"]["init_params_"]["kwargs"]["model_kwargs"]["efficiency_mode"]

'scaled'

In [15]:
ckpt["model_state_dict"]["pyro_param_store"]["params"]

{'AutoGuideList.0.locs.element_effects': Parameter containing:
 tensor([ 0.1458, -0.0365, -0.0277,  ..., -0.0100,  0.0954,  0.0162],
        requires_grad=True),
 'AutoGuideList.0.scales.element_effects': Parameter containing:
 tensor([-2.8877, -2.3072, -1.9684,  ..., -2.6081, -2.5961, -3.0503],
        requires_grad=True),
 'AutoGuideList.0.locs.guide_efficacy': Parameter containing:
 tensor([ 1.9854,  1.7796,  2.0366,  ...,  1.7227,  1.4099, -1.1098],
        requires_grad=True),
 'AutoGuideList.0.scales.guide_efficacy': Parameter containing:
 tensor([ 0.8682,  0.6898,  0.8343,  ...,  0.7251,  0.8241, -1.7325],
        requires_grad=True),
 'AutoGuideList.0.locs.log_gene_mean': Parameter containing:
 tensor([-4.5592, -3.4546, -0.1806,  ...,  0.2559, -2.9834, -4.2266],
        requires_grad=True),
 'AutoGuideList.0.scales.log_gene_mean': Parameter containing:
 tensor([-4.0279, -4.8160, -5.4490,  ..., -5.4376, -4.6008, -3.8033],
        requires_grad=True),
 'AutoGuideList.0.locs.log_g

## Test Simulate from trained data

In [3]:
# record properties from the original real-world data
n_cells_origin = 193951
n_genes_origin = 13135
n_elements_origin = 6598

print(f"In the original data, there are {n_cells_origin} cells, {n_genes_origin} genes, and {n_elements_origin} tested elements.")

In the original data, there are 193951 cells, 13135 genes, and 6598 tested elements.


In [6]:
# set fixed parameters for Gasperini at-scale data
n_genes = n_genes_origin
new_genes_idx = np.random.choice(n_genes_origin, size=n_genes, replace=False)

n_elements_pos = n_elements_origin
n_elements_ntc = round(0.05 * n_elements_pos)
n_elements = n_elements_pos + n_elements_ntc

n_grna_per_element = 4
n_grna_ntc = n_elements_ntc * n_grna_per_element
n_grna_pos = n_elements_pos * n_grna_per_element
n_grna = n_grna_ntc + n_grna_pos

#guide_efficacy = np.array([1, 2/3, 1/3, 0] * n_elements) # for method comparison: set fixed guide efficacy
#guide_efficacy = np.random.choice(guide_efficacy_real, size=n_elements * n_grna_per_element, replace=True) # for optimal experiment design: sample from estimated efficacy
guide_efficacy = np.random.beta(a=5, b=2 , size=n_elements * n_grna_per_element) # set efficiency to Beta(5,2) to mimic the case of a strong pert effect

read_depth_adjust_factor = 1  # fix read depth for now. 


In [7]:
lfc = np.random.normal(loc = -2, scale = 1, size= n_genes)  # for method comparison: set lfc to be a fixed distribution
#lfc = np.random.choice(lfc_real, size=n_genes, replace=True) # for optimal experiment design: sample from estimated lfc

element_gene_map = np.zeros((n_elements_pos, n_genes), dtype=int)  # create element-gene targeting matrix

affected_gene_idx = np.random.choice(n_genes, size=n_elements_pos, replace=False)  # Randomly select genes targeted by each element
element_gene_map[np.arange(n_elements_pos), affected_gene_idx] = 1

element_by_gene_lfc_pos = lfc * element_gene_map
element_by_gene_lfc_ntc = np.zeros((n_elements_ntc,n_genes))
element_by_gene_lfc = np.vstack((element_by_gene_lfc_pos, element_by_gene_lfc_ntc))
print(f"element_by_gene_lfc shape: {element_by_gene_lfc.shape}")


element_by_gene_lfc shape: (6928, 13135)


In [8]:
# some parameters we need to grid seach through but fix for now
moi = 30

n_cells_per_element = 10
n_cells_per_guide = n_cells_per_element // n_grna_per_element

n_cells = n_cells_per_guide * n_grna // moi
subset_indices=np.random.choice(n_cells_origin, size=n_cells, replace=False)

In [9]:
# compute grna count matrices 
#n_cells_per_guide = round(np.mean(np.sum(mdata_train["grna"].X, axis = 0)))
pert_rate = n_cells_per_guide / n_cells

#grna_counts = sparse_random(n_cells, n_grna, density=pert_rate, format="csr", dtype=np.float32)
#grna_counts.data[:] = 1

#grna_counts = np.random.binomial(1, pert_rate, size=(n_cells, n_grna)).astype(np.float32)

chunk_size = 10000  # Process 10,000 cells at a time
num_batches = n_cells // chunk_size + (n_cells % chunk_size != 0)

grna_batches = []
for i in range(num_batches):
    print(f"Processing batch {i+1}/{num_batches}...")
    batch_size_actual = min(chunk_size, n_cells - i * chunk_size)
    grna_batch = sparse_random(batch_size_actual, n_grna, density=pert_rate, format="csr", dtype=np.float32)
    grna_batch.data[:] = 1  # Convert to binary
    grna_batches.append(grna_batch)

# Combine all batches into one matrix
grna_counts = vstack(grna_batches)

print(f"gRNA count matrix have shape {grna_counts.shape}")
print(f"Number of total gRNAs is {grna_counts.nnz}")  # Number of nonzero elements
print(f"On average, each gRNA is present in {n_cells_per_guide} cells, and {pert_rate*n_grna} gRNA is present in each cell.")


Processing batch 1/3...
Processing batch 2/3...
Processing batch 3/3...
gRNA count matrix have shape (23093, 27712)
Number of total gRNAs is 692799
On average, each gRNA is present in 25 cells, and 30.000433031654616 gRNA is present in each cell.


In [10]:
# get guide-element matching matrics
guide_by_element = np.zeros((n_grna, n_elements))

for j in range(n_elements):
    start_row = n_grna_per_element * j
    end_row = start_row + n_grna_per_element
    if end_row > n_grna:
        break  # break the loop if the end_row exceeds the matrix size
    guide_by_element[start_row:end_row, j] = 1.0
    
guide_by_element = guide_by_element.astype(np.float32)
print(f"guide_by_element have shape {guide_by_element.shape}")
guide_by_element

guide_by_element have shape (27712, 6928)


array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.]], dtype=float32)

In [11]:
# Configuration:
chunk_size = 10000  # Adjust as needed to fit your GPU memory
num_cells = len(subset_indices)

# Calculate number of chunks:
num_chunks = int(np.ceil(num_cells / chunk_size))

# Track global set of all existing observation names
existing_obs_names = set()

# Placeholder list for results:
simulated_chunks = []

# Iterate over chunks
for i in range(num_chunks):
    print(f'Processing chunk {i+1}/{num_chunks}')
    
    chunk_indices = subset_indices[i * chunk_size : (i + 1) * chunk_size]   # Extract indices for current chunk
    grna_counts_chunk = grna_counts[i * chunk_size : (i + 1) * chunk_size].toarray()  # Extract corresponding grna_counts rows (sparse -> dense conversion per chunk)

    # Move to device explicitly (use CPU if needed to save GPU memory)
    #guide_obs_chunk = torch.tensor(grna_counts_chunk, dtype=torch.float32)

    # Run simulation for this chunk
    mdata_chunk = perturbo.simulation.simulate_data_from_trained_model(
        model_dir,
        guide_obs=grna_counts_chunk,
        cell_indices=chunk_indices,
        guide_by_element=guide_by_element,
        element_by_gene_lfc=element_by_gene_lfc,
        guide_efficacy=guide_efficacy,
        read_depth_adjust_factor=read_depth_adjust_factor,
        gene_indices=new_genes_idx,
        module_init_kwargs={"efficiency_mode": "mixture_high_moi"},
        accelerator='cpu'  # Force CPU computation to avoid GPU OOM
    )
    
    # Explicitly update modality-specific obs_names
    new_obs_names = []
    for obs_name in mdata_chunk.obs_names:
        if obs_name in existing_obs_names:
            obs_name = f"{obs_name}_{i+1}"
        new_obs_names.append(obs_name)

    # Update obs_names for MuData (via individual modalities)
    mdata_chunk.obs.index = new_obs_names
    for modality in mdata_chunk.mod:
        mdata_chunk.mod[modality].obs.index = new_obs_names

    # Update global obs_names set
    existing_obs_names.update(new_obs_names)

    # Store chunk results
    simulated_chunks.append(mdata_chunk)

    # Clear GPU cache if needed (if GPU computation is required later)
    torch.cuda.empty_cache()



Processing chunk 1/3


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("v

Processing chunk 2/3


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("v

Processing chunk 3/3


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_v

In [14]:
def hierarchical_concat(chunk_list):
    """Concatenate MuData objects progressively in pairs to avoid slowdowns."""
    while len(chunk_list) > 1:
        new_chunk_list = []
        for i in range(0, len(chunk_list), 2):
            if i + 1 < len(chunk_list):
                merged = md.concat([chunk_list[i], chunk_list[i + 1]])
                new_chunk_list.append(merged)
            else:
                new_chunk_list.append(chunk_list[i])
        chunk_list = new_chunk_list
        print(f"Intermediate concatenation: {len(chunk_list)} chunks remaining")
    return chunk_list[0]

# Use hierarchical_concat instead of direct concat:
mdata_simu = hierarchical_concat(simulated_chunks)

print(mdata_simu)

/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


Intermediate concatenation: 2 chunks remaining
Intermediate concatenation: 1 chunks remaining
MuData object with n_obs × n_vars = 23093 × 40847
  2 modalities
    grna:	23093 x 27712
    rna:	23093 x 13135
      obs:	'cell', 'sample', 'total_umis', 'Size_Factor', 'read_count', 'umi_count', 'proportion', 'guide_count', 'id', 'prep_batch', 'within_batch_chip', 'within_chip_lane', 'percent_mito', 'log1p_guide_count', 'log1p_gene_count', '_library_size', '_size_factor', '_ind_x', '_scvi_batch'


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


## Test Loading model with tiny MuData (Gasperini)

In [15]:
real_data_dir = "save_model/model_no_data"  # run on cluster
#real_data_dir = "C:\\Users\\JiaqiLu\\OneDrive - Helmholtz Zentrum München\\Documents\\PhD@Helmholtz\\proj0.5_ProbGen\\code\\PerTurbo\\power_tests\\data_simulation_from_posterior\\from_real_data"  # run locally
mdata_tiny = md.read_h5mu(f'{real_data_dir}/mdata_tiny.h5mu')
model = perturbo.PERTURBO.load(model_dir, adata=mdata_tiny)

model

/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


INFO     File save_model/model_no_data_with_libsize/model.pt already downloaded                                    


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
Continuous covariate 'percent_mito' has standard deviation 0.0175. Consider z-scoring.
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 I

Training:   0%|          | 0/1 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1` reached.


MyPyroModel Model with params:
n_batch: 2
n_cells: 193951
n_extra_continuous_covs: 2
n_perturbations: 13189
n_targeted_elements: 6598
n_tested_elements: 6598
n_vars: 13135

Training status: Trained

In [6]:
# record properties from the original real-world data
n_cells_origin = 193951
n_genes_origin = 13135
n_elements_origin = 6598

print(f"In the original data, there are {n_cells_origin} cells, {n_genes_origin} genes, and {n_elements_origin} tested elements.")

In the original data, there are 193951 cells, 13135 genes, and 6598 tested elements.


In [7]:
# set fixed parameters for Gasperini at-scale data
n_genes = n_genes_origin
new_genes_idx = np.random.choice(n_genes_origin, size=n_genes, replace=False)

n_elements_pos = n_elements_origin
n_elements_ntc = round(0.05 * n_elements_pos)
n_elements = n_elements_pos + n_elements_ntc

n_grna_per_element = 4
n_grna_ntc = n_elements_ntc * n_grna_per_element
n_grna_pos = n_elements_pos * n_grna_per_element
n_grna = n_grna_ntc + n_grna_pos

#guide_efficacy = np.array([1, 2/3, 1/3, 0] * n_elements) # for method comparison: set fixed guide efficacy
#guide_efficacy = np.random.choice(guide_efficacy_real, size=n_elements * n_grna_per_element, replace=True) # for optimal experiment design: sample from estimated efficacy
guide_efficacy = np.random.beta(a=5, b=2 , size=n_elements * n_grna_per_element) # set efficiency to Beta(5,2) to mimic the case of a strong pert effect

read_depth_adjust_factor = 1  # fix read depth for now. 


In [8]:
lfc = np.random.normal(loc = -2, scale = 1, size= n_genes)  # for method comparison: set lfc to be a fixed distribution
#lfc = np.random.choice(lfc_real, size=n_genes, replace=True) # for optimal experiment design: sample from estimated lfc

element_gene_map = np.zeros((n_elements_pos, n_genes), dtype=int)  # create element-gene targeting matrix

affected_gene_idx = np.random.choice(n_genes, size=n_elements_pos, replace=False)  # Randomly select genes targeted by each element
element_gene_map[np.arange(n_elements_pos), affected_gene_idx] = 1

element_by_gene_lfc_pos = lfc * element_gene_map
element_by_gene_lfc_ntc = np.zeros((n_elements_ntc,n_genes))
element_by_gene_lfc = np.vstack((element_by_gene_lfc_pos, element_by_gene_lfc_ntc))
print(f"element_by_gene_lfc shape: {element_by_gene_lfc.shape}")


element_by_gene_lfc shape: (6928, 13135)


In [9]:
# some parameters we need to grid seach through but fix for now
moi = 30

n_cells_per_element = 100
n_cells_per_guide = n_cells_per_element // n_grna_per_element

n_cells = n_cells_per_guide * n_grna // moi
subset_indices=np.random.choice(n_cells_origin, size=n_cells, replace=False)

In [10]:
# compute grna count matrices 
#n_cells_per_guide = round(np.mean(np.sum(mdata_train["grna"].X, axis = 0)))
pert_rate = n_cells_per_guide / n_cells

#grna_counts = sparse_random(n_cells, n_grna, density=pert_rate, format="csr", dtype=np.float32)
#grna_counts.data[:] = 1

#grna_counts = np.random.binomial(1, pert_rate, size=(n_cells, n_grna)).astype(np.float32)

chunk_size = 10000  # Process 10,000 cells at a time
num_batches = n_cells // chunk_size + (n_cells % chunk_size != 0)

grna_batches = []
for i in range(num_batches):
    print(f"Processing batch {i+1}/{num_batches}...")
    batch_size_actual = min(chunk_size, n_cells - i * chunk_size)
    grna_batch = sparse_random(batch_size_actual, n_grna, density=pert_rate, format="csr", dtype=np.float32)
    grna_batch.data[:] = 1  # Convert to binary
    grna_batches.append(grna_batch)

# Combine all batches into one matrix
grna_counts = vstack(grna_batches)

print(f"gRNA count matrix have shape {grna_counts.shape}")
print(f"Number of total gRNAs is {grna_counts.nnz}")  # Number of nonzero elements
print(f"On average, each gRNA is present in {n_cells_per_guide} cells, and {pert_rate*n_grna} gRNA is present in each cell.")


Processing batch 1/3...
Processing batch 2/3...
Processing batch 3/3...
gRNA count matrix have shape (23093, 27712)
Number of total gRNAs is 692799
On average, each gRNA is present in 25 cells, and 30.000433031654616 gRNA is present in each cell.


In [11]:
# get guide-element matching matrics
guide_by_element = np.zeros((n_grna, n_elements))

for j in range(n_elements):
    start_row = n_grna_per_element * j
    end_row = start_row + n_grna_per_element
    if end_row > n_grna:
        break  # break the loop if the end_row exceeds the matrix size
    guide_by_element[start_row:end_row, j] = 1
    
guide_by_element = guide_by_element.astype(np.float32)
print(f"guide_by_element have shape {guide_by_element.shape}")
guide_by_element

guide_by_element have shape (27712, 6928)


array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.]], dtype=float32)

In [12]:
# Configuration:
chunk_size = 10000  # Adjust as needed to fit your GPU memory
num_cells = len(subset_indices)

# Calculate number of chunks:
num_chunks = int(np.ceil(num_cells / chunk_size))

# Track global set of all existing observation names
existing_obs_names = set()

# Placeholder list for results:
simulated_chunks = []

# Iterate over chunks
for i in range(num_chunks):
    print(f'Processing chunk {i+1}/{num_chunks}')
    
    chunk_indices = subset_indices[i * chunk_size : (i + 1) * chunk_size]   # Extract indices for current chunk
    grna_counts_chunk = grna_counts[i * chunk_size : (i + 1) * chunk_size].toarray()  # Extract corresponding grna_counts rows (sparse -> dense conversion per chunk)

    # Move to device explicitly (use CPU if needed to save GPU memory)
    #guide_obs_chunk = torch.tensor(grna_counts_chunk, dtype=torch.float32)

    # Run simulation for this chunk
    mdata_chunk = perturbo.simulation.simulate_data_from_trained_model(
        model,
        guide_obs=grna_counts_chunk,
        cell_indices=chunk_indices,
        guide_by_element=guide_by_element,
        element_by_gene_lfc=element_by_gene_lfc,
        guide_efficacy=guide_efficacy,
        read_depth_adjust_factor=read_depth_adjust_factor,
        gene_indices=new_genes_idx,
        module_init_kwargs={"efficiency_mode": "mixture_high_moi"},
        accelerator='cpu'  # Force CPU computation to avoid GPU OOM
    )
    
    # Explicitly update modality-specific obs_names
    new_obs_names = []
    for obs_name in mdata_chunk.obs_names:
        if obs_name in existing_obs_names:
            obs_name = f"{obs_name}_{i+1}"
        new_obs_names.append(obs_name)

    # Update obs_names for MuData (via individual modalities)
    mdata_chunk.obs.index = new_obs_names
    for modality in mdata_chunk.mod:
        mdata_chunk.mod[modality].obs.index = new_obs_names

    # Update global obs_names set
    existing_obs_names.update(new_obs_names)

    # Store chunk results
    simulated_chunks.append(mdata_chunk)

    # Clear GPU cache if needed (if GPU computation is required later)
    torch.cuda.empty_cache()



Processing chunk 1/3


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("v

Processing chunk 2/3


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("v

Processing chunk 3/3


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_v

In [13]:
def hierarchical_concat(chunk_list):
    """Concatenate MuData objects progressively in pairs to avoid slowdowns."""
    while len(chunk_list) > 1:
        new_chunk_list = []
        for i in range(0, len(chunk_list), 2):
            if i + 1 < len(chunk_list):
                merged = md.concat([chunk_list[i], chunk_list[i + 1]])
                new_chunk_list.append(merged)
            else:
                new_chunk_list.append(chunk_list[i])
        chunk_list = new_chunk_list
        print(f"Intermediate concatenation: {len(chunk_list)} chunks remaining")
    return chunk_list[0]

# Use hierarchical_concat instead of direct concat:
mdata_simu = hierarchical_concat(simulated_chunks)

print(mdata_simu)

/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


Intermediate concatenation: 2 chunks remaining
Intermediate concatenation: 1 chunks remaining
MuData object with n_obs × n_vars = 23093 × 40847
  2 modalities
    grna:	23093 x 27712
    rna:	23093 x 13135
      obs:	'cell', 'sample', 'total_umis', 'Size_Factor', 'read_count', 'umi_count', 'proportion', 'guide_count', 'id', 'prep_batch', 'within_batch_chip', 'within_chip_lane', 'percent_mito', 'log1p_guide_count', 'log1p_gene_count', '_library_size', '_size_factor', '_ind_x', '_scvi_batch'


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


## Test Loading model with tiny MuData (Weissman)

In [3]:
model_dir = Path("../Weissman_ess/save_model/model_mixture")
print(f"Model is loaded from path: {model_dir}")

ckpt = torch.load(model_dir / "model.pt", map_location="cpu")
registry = ckpt["attr_dict"]["registry_"]
print(f"Registry {registry} elements.")

Model is loaded from path: ../Weissman_ess/save_model/model_mixture
Registry {'scvi_version': '1.4.0.post1', 'model_name': 'PERTURBO', 'setup_args': {'rna_layer': None, 'perturbation_layer': None, 'batch_key': 'gem_group', 'gene_by_element_key': 'gene_by_element', 'rna_element_uns_key': None, 'guide_element_uns_key': None, 'guide_by_element_key': 'guide_by_element', 'library_size_key': 'UMI_count', 'size_factor_key': None, 'gene_mean_key': None, 'continuous_covariates_keys': None, 'categorical_covariates_keys': None, 'modalities': {'rna_layer': 'rna', 'perturbation_layer': 'grna'}}, 'field_registries': defaultdict(<class 'dict'>, {'ind_x': {'data_registry': {'attr_name': 'obs', 'attr_key': '_ind_x', 'mod_key': 'rna'}, 'state_registry': {}, 'summary_stats': {}}, 'batch': {'data_registry': {'attr_name': 'obs', 'attr_key': '_scvi_batch', 'mod_key': 'rna'}, 'state_registry': {'categorical_mapping': array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16

In [4]:
real_data_dir = "../Weissman_ess/save_model/model_mixture"  # run on cluster
#real_data_dir = "C:\\Users\\JiaqiLu\\OneDrive - Helmholtz Zentrum München\\Documents\\PhD@Helmholtz\\proj0.5_ProbGen\\code\\PerTurbo\\power_tests\\data_simulation_from_posterior\\from_real_data"  # run locally
mdata_tiny = md.read_h5mu(f'{real_data_dir}/mdata_tiny.h5mu')
model = perturbo.PERTURBO.load(model_dir, adata=mdata_tiny)

model

/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


INFO     File ../Weissman_ess/save_model/model_mixture/model.pt already downloaded                                 


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib

Training:   0%|          | 0/1 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1` reached.


MyPyroModel Model with params:
n_batch: 267
n_cells: 1989578
n_perturbations: 11187
n_targeted_elements: 9866
n_tested_elements: 9866
n_vars: 8248

Training status: Trained

In [5]:
# record properties from the original real-world data
n_cells_origin = 1989578
n_genes_origin = 8248
n_elements_origin = 9866

print(f"In the original data, there are {n_cells_origin} cells, {n_genes_origin} genes, and {n_elements_origin} tested elements.")

In the original data, there are 1989578 cells, 8248 genes, and 9866 tested elements.


In [8]:
# set fixed parameters for Gasperini at-scale data
n_genes = n_genes_origin
new_genes_idx = np.random.choice(n_genes_origin, size=n_genes, replace=False)

n_elements_pos = min(n_elements_origin, n_genes)
n_elements_ntc = round(0.05 * n_elements_pos)
n_elements = n_elements_pos + n_elements_ntc

n_grna_per_element = 1
n_grna_ntc = n_elements_ntc * n_grna_per_element
n_grna_pos = n_elements_pos * n_grna_per_element
n_grna = n_grna_ntc + n_grna_pos

#guide_efficacy = np.array([1, 2/3, 1/3, 0] * n_elements) # for method comparison: set fixed guide efficacy
#guide_efficacy = np.random.choice(guide_efficacy_real, size=n_elements * n_grna_per_element, replace=True) # for optimal experiment design: sample from estimated efficacy
guide_efficacy = np.random.beta(a=5, b=2 , size=n_elements * n_grna_per_element) # set efficiency to Beta(5,2) to mimic the case of a strong pert effect

read_depth_adjust_factor = 1  # fix read depth for now. 


In [9]:
lfc = np.random.normal(loc = -2, scale = 1, size= n_genes)  # for method comparison: set lfc to be a fixed distribution
#lfc = np.random.choice(lfc_real, size=n_genes, replace=True) # for optimal experiment design: sample from estimated lfc

element_gene_map = np.zeros((n_elements_pos, n_genes), dtype=int)  # create element-gene targeting matrix

affected_gene_idx = np.random.choice(n_genes, size=n_elements_pos, replace=False)  # Randomly select genes targeted by each element
element_gene_map[np.arange(n_elements_pos), affected_gene_idx] = 1

element_by_gene_lfc_pos = lfc * element_gene_map
element_by_gene_lfc_ntc = np.zeros((n_elements_ntc,n_genes))
element_by_gene_lfc = np.vstack((element_by_gene_lfc_pos, element_by_gene_lfc_ntc))
print(f"element_by_gene_lfc shape: {element_by_gene_lfc.shape}")


element_by_gene_lfc shape: (8660, 8248)


In [144]:
# some parameters we need to grid seach through but fix for now
moi = 1

n_cells_per_element = 20
n_cells_per_guide = n_cells_per_element // n_grna_per_element

n_cells = n_cells_per_guide * n_grna // moi
subset_indices=np.random.choice(n_cells_origin, size=n_cells, replace=False)

In [102]:
def fit_negative_binomial(data, fallback='poisson'):
    """
    Estimate Negative Binomial parameters (n, p) using the method of moments.

    The np.random.negative_binomial(n, p) function uses a parameterization where:
      - n is the number of successes,
      - p is the success probability,
      - the returned value is the number of failures before achieving n successes.

    Mean = n*(1-p)/p and Variance = n*(1-p)/p^2.

    Parameters:
      data : array-like of non-negative integers
    Returns:
      n, p : estimated parameters
    """
    m = np.mean(data)
    v = np.var(data, ddof=1)

    if v <= m:
        if fallback == 'poisson':
            print("Warning: Variance ≤ mean. Falling back to Poisson approximation.")
            # Poisson-like behavior: set very high dispersion (small p, large n)
            p = 1e-5
            n = m * p / (1 - p)
            return n, p
        else:
            raise ValueError("Data variance must be greater than the mean for a valid Negative Binomial fit.")

    p = m / v
    n = m**2 / (v - m)

    return n, p

In [137]:
print(f"Simulating with n_cells_per_guide: {n_cells_per_guide}, n_cells: {n_cells}, read_scale: {read_depth_adjust_factor},, MOI: {moi}.")

#n_cells_per_guide_real = mdata_real["grna"].X.sum(axis=0).A1
summ = np.load("../Weissman_ess/save_model/model_mixture/mdata_summary_weissman.npz")
n_cells_per_guide_real = summ["n_cells_per_guide_real"]
#n_cells_per_guide_real_scaled = (n_cells_per_guide / n_cells_per_guide_real.mean()) * n_cells_per_guide_real
#n, p = fit_negative_binomial(np.round(n_cells_per_guide_real_scaled))
n, p = fit_negative_binomial(np.round(n_cells_per_guide_real))
print("Fitted parameters (n, p):", n, p)

# Sample a list of n_cells_per_guide
n_cells_per_guide_list = np.round(nbinom.rvs(n, p, size=n_grna) * (n_cells_per_guide / n_cells_per_guide_real.mean())).astype(int)


Simulating with n_cells_per_guide: 10, n_cells: 86600, read_scale: 1,, MOI: 1.
Fitted parameters (n, p): 3.129291306771912 0.017291136


In [145]:
def fit_nb_moments(x, eps=1e-8):
    # method-of-moments NB fit (SciPy's param: n=size=r, p=success prob)
    mu = np.mean(x)
    var = np.var(x, ddof=1)
    if var <= mu + eps:   # near/under-Poisson: fall back to Poisson-ish
        return np.inf, mu  # mark as Poisson with mean mu
    r = mu**2 / (var - mu)  # size/dispersion
    return r, mu
def sample_cells_per_guide(real_counts, mu_target, n_grna, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    r, mu_real = fit_nb_moments(real_counts)
    if np.isinf(r):  # Poisson fallback
        return rng.poisson(lam=mu_target, size=n_grna).astype(int)
    p_target = r / (r + mu_target)
    # SciPy: nbinom.rvs(n=r, p=p) returns number of failures before n successes
    return nbinom.rvs(r, p_target, size=n_grna, random_state=rng).astype(int)

In [147]:
n_cells_per_guide_list = sample_cells_per_guide(n_cells_per_guide_real, n_cells_per_guide, n_grna)
n_cells_per_guide_list

array([28, 19,  0, 10,  6, 34, 50, 25, 24, 18])

In [148]:
# compute grna count matrices 
#n_cells_per_guide = round(np.mean(np.sum(mdata_train["grna"].X, axis = 0)))
pert_rate = n_cells_per_guide_list / n_cells

#grna_counts = sparse_random(n_cells, n_grna, density=pert_rate, format="csr", dtype=np.float32)
#grna_counts.data[:] = 1

#grna_counts = np.random.binomial(1, pert_rate, size=(n_cells, n_grna)).astype(np.float32)

chunk_size = 10000  # Process 10,000 cells at a time
num_batches = n_cells // chunk_size + (n_cells % chunk_size != 0)

rows, cols = [], []
perm = np.random.permutation(n_cells)
ptr  = 0
for g, k in enumerate(n_cells_per_guide_list):
    if ptr >= n_cells:
        break                      # no cells left
    take = min(k, n_cells - ptr)   # clip the remainder
    rows.append(perm[ptr:ptr+take])
    cols.append(np.full(take, g, dtype=np.int32))
    ptr += take
rows = np.concatenate(rows)
cols = np.concatenate(cols)
data = np.ones_like(rows, dtype=np.float32)
grna_counts = csr_matrix((data, (rows, cols)),
                        shape=(n_cells, n_grna), dtype=np.float32)


print(f"gRNA count matrix have shape {grna_counts.shape}")
print(f"Number of total gRNAs is {grna_counts.nnz}")  # Number of nonzero elements
print(f"The average number of gRNAs per cell is {grna_counts.sum(axis=0).mean()}")
print(f"On average, each gRNA is present in {n_cells_per_guide_list.mean()} cells, and {pert_rate.mean()*n_grna} gRNA is present in each cell.")


gRNA count matrix have shape (173200, 8660)
Number of total gRNAs is 173200
The average number of gRNAs per cell is 20.0
On average, each gRNA is present in 20.064665127020785 cells, and 1.003233256351039 gRNA is present in each cell.


In [149]:
# get guide-element matching matrics
guide_by_element = np.zeros((n_grna, n_elements))

for j in range(n_elements):
    start_row = n_grna_per_element * j
    end_row = start_row + n_grna_per_element
    if end_row > n_grna:
        break  # break the loop if the end_row exceeds the matrix size
    guide_by_element[start_row:end_row, j] = 1
    
guide_by_element = guide_by_element.astype(np.float32)
print(f"guide_by_element have shape {guide_by_element.shape}")
guide_by_element

guide_by_element have shape (8660, 8660)


array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], dtype=float32)

In [150]:
# Configuration:
chunk_size = 10000  # Adjust as needed to fit your GPU memory
num_cells = len(subset_indices)

# Calculate number of chunks:
num_chunks = int(np.ceil(num_cells / chunk_size))

# Track global set of all existing observation names
existing_obs_names = set()

# Placeholder list for results:
simulated_chunks = []

# Iterate over chunks
for i in range(num_chunks):
    print(f'Processing chunk {i+1}/{num_chunks}')
    
    grna_counts_chunk = grna_counts[i * chunk_size : (i + 1) * chunk_size].toarray()  # Extract corresponding grna_counts rows (sparse -> dense conversion per chunk)
    chunk_indices = subset_indices[i * chunk_size : (i + 1) * chunk_size]   # Extract indices for current chunk
    #chunk_indices = np.random.choice(n_cells_origin, size=grna_counts_chunk.shape[0], replace=False)   # Extract indices for current chunk
    
    # Move to device explicitly (use CPU if needed to save GPU memory)
    #guide_obs_chunk = torch.tensor(grna_counts_chunk, dtype=torch.float32)

    # Run simulation for this chunk
    mdata_chunk = perturbo.simulation.simulate_data_from_trained_model(
        model,
        guide_obs=grna_counts_chunk,
        cell_indices=chunk_indices,
        guide_by_element=guide_by_element,
        element_by_gene_lfc=element_by_gene_lfc,
        guide_efficacy=guide_efficacy,
        read_depth_adjust_factor=read_depth_adjust_factor,
        gene_indices=new_genes_idx,
        module_init_kwargs={"efficiency_mode": "mixture_high_moi"},
        accelerator='cpu'  # Force CPU computation to avoid GPU OOM
    )
    
    # Explicitly update modality-specific obs_names
    new_obs_names = []
    for obs_name in mdata_chunk.obs_names:
        if obs_name in existing_obs_names:
            obs_name = f"{obs_name}_{i+1}"
        new_obs_names.append(obs_name)

    # Update obs_names for MuData (via individual modalities)
    mdata_chunk.obs.index = new_obs_names
    for modality in mdata_chunk.mod:
        mdata_chunk.mod[modality].obs.index = new_obs_names

    # Update global obs_names set
    existing_obs_names.update(new_obs_names)

    # Store chunk results
    simulated_chunks.append(mdata_chunk)

    # Clear GPU cache if needed (if GPU computation is required later)
    torch.cuda.empty_cache()



Processing chunk 1/18


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("v

Processing chunk 2/18


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_v

Processing chunk 3/18


...

/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_v

Processing chunk 17/18


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_v

Processing chunk 18/18


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_v

In [ ]:
def hierarchical_concat(chunk_list):
    """Concatenate MuData objects progressively in pairs to avoid slowdowns."""
    while len(chunk_list) > 1:
        new_chunk_list = []
        for i in range(0, len(chunk_list), 2):
            if i + 1 < len(chunk_list):
                merged = md.concat([chunk_list[i], chunk_list[i + 1]])
                new_chunk_list.append(merged)
            else:
                new_chunk_list.append(chunk_list[i])
        chunk_list = new_chunk_list
        print(f"Intermediate concatenation: {len(chunk_list)} chunks remaining")
    return chunk_list[0]

# Use hierarchical_concat instead of direct concat:
mdata_simu = hierarchical_concat(simulated_chunks)

print(mdata_simu)

/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/muda

Intermediate concatenation: 9 chunks remaining


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/muda

In [ ]:
## add element_tested
tested_elements_pos = element_gene_map

n_ntc_pairs = n_elements_pos  # set the same number of positive and negative pairs
test_rate = n_ntc_pairs / (n_elements_ntc * n_genes)
tested_elements_ntc = np.random.binomial(1, test_rate, size=(n_elements_ntc, n_genes)).astype(np.float32)

tested_elements = np.vstack((tested_elements_pos, tested_elements_ntc))

mdata_simu["rna"].varm["gene_by_element"] = tested_elements.T
mdata_simu["rna"].varm["lfc"] = simulated_chunks[0].mod["rna"].varm["lfc"]
mdata_simu["rna"].var = simulated_chunks[0].mod["rna"].var
mdata_simu["grna"].varm["guide_by_element"] = simulated_chunks[0].mod["grna"].varm["guide_by_element"]
mdata_simu.update()

mdata_simu

Train

In [127]:
selected_cells = mdata_simu["rna"].X.sum(axis=1)>0
mdata_filtered = mdata_simu[selected_cells,:].copy()


/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [128]:
lr = 0.01
batch_size = 4096
max_steps = 200
likelihood = "nb"

alpha = 0.01
MTmethod = "fdr"

if MTmethod == "fdr":
    MT = "BH"
elif MTmethod == "none":
    MT = "none"

In [129]:
n_elements = mdata_filtered["rna"].varm["gene_by_element"].shape[1]
n_grna_per_element = 1
n_cells = mdata_filtered["rna"].X.shape[0] 
n_grna = mdata_filtered["grna"].X.shape[1]

In [130]:
(mdata_filtered["grna"].X.sum(axis=1)>1).sum()

0

In [131]:
mdata_filtered["grna"].X.sum(axis=1)

array([1., 1., 1., ..., 1., 1., 1.], dtype=float32)

In [132]:
for mod in ["rna", "grna"]:
    X = mdata_filtered.mod[mod].X
    if X.dtype != "float32":
        mdata_filtered.mod[mod].X = csr_matrix(X, dtype="float32")


In [133]:
perturbo.PERTURBO.setup_mudata(
    mdata_filtered,
    batch_key="gem_group",
    library_size_key="UMI_count",
    #gene_by_element_key='gene_by_element', # <------------ gene * element pairs
    guide_by_element_key="guide_by_element",
    #guide_element_uns_key=guide_element_uns_key,  # <------------ list of elements tested here
    #rna_element_uns_key=gene_element_uns_key,    # <------------ what is this?
    modalities={
        "rna_layer": "rna",
        "perturbation_layer": "grna",
    },
) 

INFO     Generating sequential column names                                                                        


In [134]:
model_eval = perturbo.PERTURBO(mdata_filtered, 
                                likelihood="nb", 
                                efficiency_mode = "mixture",
                                #effect_prior_dist="normal", 
                                fit_guide_efficacy = False,
                                #prior_param_dict = {"gene_disp_prior_loc": torch.ones(n_genes_origin, dtype=torch.float32) * 2},
                                #n_factors=5
                                )
pyro.clear_param_store()

In [135]:
model_eval.train(
    max_epochs=200, 
    lr=0.01, 
    batch_size=4096, 
    accelerator="gpu",
    early_stopping=True,
    early_stopping_patience=5,
    early_stopping_min_delta=1e-3,
    early_stopping_monitor="elbo_train",
)

/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:165: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/pyth ...
  rank_zero_warn(
/home/icb/jiaqi.lu/miniconda3/envs/perturbo/lib/

Training:   0%|          | 0/200 [00:00<?, ?it/s]

Monitored metric elbo_train did not improve in the last 5 records. Best score: 698195072.000. Signaling Trainer to stop.
